# 07: Label Definitions: Telling the LLM What Each Label Means (Contribution F)

**Why this notebook:** two earlier findings point to the same weakness:
- **Massive-I:** many intents differ by a single action word ("set" vs "query" an alarm), and the small LLM confuses them.
- **Outlier analysis (06):** factual questions ("is steph curry married") were grouped by *subject* (sports, celebrity) instead of by *function* (asking a factual question).

In both cases the LLM only sees bare label **names** like `alarm setting` or `qa`, which are short and ambiguous.

**Idea:** give every label an **explicit definition**:
```
alarm setting: The user wants to create a new alarm.  (keywords: wake me up, set an alarm, alarm for)
```
The definition describes **what the user wants**, so the model is told that the action or function decides the category, not just the topic.

**How we keep the comparison fair:**
1. We reuse the **same merged label list** as notebook 02 (same threshold). Only the prompt changes.
2. The **LLM writes the definitions itself**, so no true labels are used. For each label it sees a few texts it had already assigned to that label in notebook 02, and the names of the other labels, so it can explain what makes this label different.
3. Classification, rescue (Fix C) and evaluation are **exactly as in notebook 02**.

So the result is directly comparable to "A+B+C (merge + rescue)", and the only difference is the definitions. We call this variant **F**.

**Cost:** about one short LLM call per label (36–61) to write the definitions, plus one classification run. The prompts are longer, so classification is slower than in notebook 02 (roughly 1.5–2.5×).

**Technical note:** with definitions, each prompt can exceed 2,000 tokens, and Ollama silently cuts prompts longer than its default context window. This notebook therefore calls Ollama's own API with a larger context window (8,192 tokens) and reports the longest prompt, so you can check nothing was cut.

## 0. Setup
Choose the dataset and model. `THRESHOLD` must be the one chosen in notebook 02 for this dataset and model (Massive-D + Qwen 2.5: 0.5; Massive-I + Qwen 2.5: 0.4; Massive-D + Qwen3: 0.6).

In [1]:
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import json, time, math, random, importlib, io, contextlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from types import SimpleNamespace

DATA = "massive_intent"
MODEL = "qwen2.5:7b"
THRESHOLD = 0.4          # the threshold chosen in notebook 02 for this dataset and model
SIZE = "small"
N_EXAMPLES = 5           # example texts shown per label when writing its definition
METHOD_F = "F (A+B+C + label definitions)"
OURS = "A+B+C (merge + rescue)"

OUT = "./generated_labels" if MODEL == "qwen2.5:7b" else f"./generated_labels/{MODEL.replace(':', '_')}"
def out_file(name):
    return f"{OUT}/{DATA}_{SIZE}_{name}"

import evaluate as ev
import given_label_classification as glc   # used for answer parsing and matching only

# The prompts with definitions are long. Ollama silently cuts prompts longer than its context window
# (2,048-4,096 tokens by default), so we call its own API with a larger window (num_ctx).
import urllib.request
NUM_CTX = 8192
def chat_long(prompt):
    body = {"model": MODEL, "stream": False, "format": "json",
            "options": {"temperature": 0, "num_ctx": NUM_CTX},
            "messages": [{"role": "system", "content": "You are a helpful assistant designed to output JSON."},
                         {"role": "user", "content": prompt}]}
    if "qwen3" in MODEL.lower():
        body["think"] = False
    req = urllib.request.Request("http://localhost:11434/api/chat", data=json.dumps(body).encode("utf-8"),
                                 headers={"Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(req, timeout=600) as r:
            out = json.loads(r.read().decode("utf-8"))
        return out["message"]["content"], out.get("prompt_eval_count")
    except Exception as e:
        print("LLM call failed:", e)
        return None, None

gold = ev.load_data("./dataset/", DATA, SIZE == "large")
texts = [g["input"] for g in gold]
y_true_raw = ev.get_labels(gold)
with open(out_file(f"improved_labels_t{THRESHOLD}.json"), encoding="utf-8") as f:
    labels = json.load(f)
labels = list(dict.fromkeys(labels))      # same list as notebook 02, duplicates removed
print(f"{DATA} / {MODEL} / t={THRESHOLD}: {len(texts)} texts, {len(labels)} labels")

massive_intent / qwen2.5:7b / t=0.4: 2974 texts, 61 labels


## 1. Rebuild notebook 02's predictions (for examples and for comparison)
We load the answers saved by notebook 02 and apply the same strict matching + rescue. This gives:
- **examples for each label** (texts the LLM itself put there), used to write the definitions;
- **the A+B+C predictions**, the comparison point for F.

In [2]:
from sentence_transformers import SentenceTransformer
minilm = SentenceTransformer("all-MiniLM-L6-v2")
def embed(items):
    return minilm.encode([str(x).replace("_", " ") for x in items], normalize_embeddings=True)

def answer_text(raw):
    try:
        obj = glc.parse_json(raw)
        vals = [v for v in obj.values() if isinstance(v, str)]
        return vals[0] if vals else str(raw)
    except Exception:
        return str(raw or "")

def to_predictions(raw_answers, label_list):
    strict = [glc.answer_process(r, label_list) if r else "Unsuccessful" for r in raw_answers]
    failed = [i for i, p in enumerate(strict) if p == "Unsuccessful"]
    rescued = list(strict)
    if failed:
        E_lab = embed(label_list); E_ans = embed([answer_text(raw_answers[i]) for i in failed])
        for i, j in zip(failed, (E_ans @ E_lab.T).argmax(axis=1)):
            rescued[i] = label_list[j]
    return strict, rescued, failed

raw02 = {}
with open(out_file(f"improved_t{THRESHOLD}_raw.jsonl"), encoding="utf-8") as f:
    for line in f:
        r = json.loads(line); raw02[r["i"]] = r["raw"]
raw02 = [raw02[i] for i in range(len(texts))]
strict02, preds02, failed02 = to_predictions(raw02, labels)
print(f"Notebook 02 run: {len(failed02)} answers needed rescue")

random.seed(0)
examples = {}
for lab in labels:
    members = [texts[i] for i, p in enumerate(preds02) if p == lab]
    examples[lab] = random.sample(members, min(N_EXAMPLES, len(members)))
print("Labels without any example:", [l for l in labels if not examples[l]])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Notebook 02 run: 433 answers needed rescue
Labels without any example: ['email querycontact', 'qa_email']


## 2. Let the LLM write a definition for each label
For each label, the prompt contains its name, up to 5 texts assigned to it, and the other label names. We ask for:
- a **one-sentence definition** of what the user wants (the action or purpose, not just the subject);
- **3–6 keywords** typical for that label.

Definitions are saved to a file, so this step runs only once. **Read them before classifying:** if one is clearly wrong, you can fix it by hand in the file (`..._label_definitions_t{THRESHOLD}.json`) and rerun the next cells.

In [3]:
def definition_prompt(label, ex, others):
    json_example = {"definition": "The user wants to ...", "keywords": ["...", "..."]}
    p = "You are helping to describe categories for classifying short user requests to a voice assistant.\n"
    p += f"Category name: {label}\n"
    p += f"Example requests in this category: {ex}\n"
    p += f"Other categories that exist: {others}\n"
    p += ("Write ONE sentence that defines what the user wants in this category: focus on the action or purpose "
          "of the request (e.g. setting vs asking vs removing), not only on the topic, and make it clearly different "
          "from the other categories. Then give 3 to 6 typical keywords or short phrases.\n")
    p += f"Return JSON like: {json_example}"
    return p

def_path = out_file(f"label_definitions_t{THRESHOLD}.json")
defs = json.load(open(def_path, encoding="utf-8")) if os.path.exists(def_path) else {}
t0 = time.time()
for lab in labels:
    if lab in defs:
        continue
    others = [l for l in labels if l != lab]
    raw, _ = chat_long(definition_prompt(lab, examples[lab], others))
    try:
        obj = glc.parse_json(raw)
        definition = str(obj.get("definition", "")).strip()
        kws = obj.get("keywords", [])
        kws = [str(k) for k in kws] if isinstance(kws, list) else [str(kws)]
    except Exception:
        definition, kws = "", []
    defs[lab] = {"definition": definition, "keywords": kws[:6]}
    json.dump(defs, open(def_path, "w", encoding="utf-8"), indent=2, ensure_ascii=False)
time_defs = (time.time() - t0) / 60
print(f"{len(defs)} definitions ({time_defs:.1f} min this session). Saved to {def_path}")
pd.set_option("display.max_colwidth", 120); pd.set_option("display.max_rows", 200)
pd.DataFrame([{"label": l, "definition": defs[l]["definition"], "keywords": ", ".join(defs[l]["keywords"])} for l in labels])

61 definitions (3.8 min this session). Saved to ./generated_labels/massive_intent_small_label_definitions_t0.4.json


,label,definition,keywords
0,music query,"The user wants to query or interact with music-related information or services, such as playing, deleting, or recomm...","query music, play song, delete playlist, recommend playlist, current songs, upcoming music"
1,play music,The user wants to initiate or control the playback of music according to their preferences or specific requests.,"play, music, album, playlist, songs, turn on"
2,iot hue lightoff,The user wants to turn off a smart light controlled by the Hue system.,"turn off, Hue light, light off, smart light, iot hue"
3,transport ticket,The user wants to purchase or obtain a transport ticket for a specific destination.,"train ticket, buy ticket, ticket for, transport ticket"
4,email sendemail,The user wants to send an email to a specified recipient with a given message content.,"send email, email to, message content, recipient, send now"
5,news query,"The user wants to receive or be informed about current news and updates, often seeking specific topics or sources.","news, headline, update, current, specific topic, source"
6,social post,"The user wants to share personal updates, express opinions, or send messages to their social network or friends.","post to social media, send message, share update, tweet about, facebook post, send to friends"
7,cooking recipe,The user wants to receive a specific recipe for a dish or meal from the voice assistant.,"recipe, prepare, homemade, dish, meal"
8,qa stock,The user wants to inquire about the current stock price or recent changes in a specific company's stock.,"stock price, company stock, up and down, today's stocks, detail for stock"
9,email querycontact,"The user wants to inquire about or seek assistance with contacting someone via email, possibly requesting informatio...","email inquiry, contact assistance, email support, query contact, email help, contact information"


## 3. Classify every text with the definitions
Same as notebook 02 (one call per text, same model, same answer format), but the label list in the prompt now shows each label's definition and keywords. Answers are saved line by line, so the run can resume if it stops.

In [4]:
def prompt_with_definitions(sentence):
    lines = []
    for lab in labels:
        d = defs.get(lab, {})
        kw = ", ".join(d.get("keywords", []))
        lines.append(f"- {lab}: {d.get('definition', '')}" + (f" (keywords: {kw})" if kw else ""))
    json_example = {"label_name": "label"}
    p = "Given the categories below and the sentence, choose the ONE category that best matches what the user wants.\n"
    p += "Categories (name: definition):\n" + "\n".join(lines) + "\n"
    p += f"Sentence: {sentence}\n"
    p += f"Return only the category name, exactly as written above, in JSON format like: {json_example}"
    return p

from tqdm.auto import tqdm
TAG = f"definitions_t{THRESHOLD}"
raw_path = out_file(f"{TAG}_raw.jsonl")
done = {}
if os.path.exists(raw_path):
    with open(raw_path, encoding="utf-8") as f:
        for line in f:
            r = json.loads(line); done[r["i"]] = r["raw"]
t0 = time.time()
max_tokens = 0
with open(raw_path, "a", encoding="utf-8") as f:
    for i, text in enumerate(tqdm(texts, desc=TAG)):
        if i in done:
            continue
        raw, n_tokens = chat_long(prompt_with_definitions(text))
        if n_tokens:
            max_tokens = max(max_tokens, n_tokens)
        done[i] = raw
        f.write(json.dumps({"i": i, "raw": raw}, ensure_ascii=False) + "\n"); f.flush()
time_classify = (time.time() - t0) / 60
raw_def = [done[i] for i in range(len(texts))]
print(f"Done. Time this session: {time_classify:.1f} min")
print(f"Longest prompt seen by the model: {max_tokens} tokens (context window: {NUM_CTX})")
if max_tokens >= NUM_CTX:
    print("WARNING: prompts reached the context limit and may have been cut. Increase NUM_CTX and rerun.")

definitions_t0.4:   0%|          | 0/2974 [00:00<?, ?it/s]

Done. Time this session: 119.0 min
Longest prompt seen by the model: 2882 tokens (context window: 8192)


## 4. Rescue and evaluate
Same strict matching and rescue (Fix C) as notebook 02, then the same metrics. We show four rows:
- **A+B (no definitions)** and **A+B+C (no definitions)** from notebook 02's saved answers;
- **with definitions, before rescue** and **with definitions + rescue (F)**.

In [5]:
strictF, predsF, failedF = to_predictions(raw_def, labels)

def score(preds):
    with contextlib.redirect_stdout(io.StringIO()):
        yt, _ = ev.convert_label_to_ids(y_true_raw); yp, k = ev.convert_label_to_ids(preds)
        s = ev.clustering_score(yt, yp)
    return {m: round(100 * float(v), 1) for m, v in s.items()}, k

rows = []
for name, p, nfail in [("A+B, no definitions (02)", strict02, len(failed02)),
                       ("A+B+C, no definitions (02)", preds02, 0),
                       ("A+B + definitions (before rescue)", strictF, len(failedF)),
                       ("A+B+C + definitions = F", predsF, 0)]:
    s, k = score(p)
    rows.append({"variant": name, "clusters": k, **s, "failed answers": nfail})
comparison = pd.DataFrame(rows)
comparison

,variant,clusters,ACC,ARI,NMI,failed answers
0,"A+B, no definitions (02)",51,42.1,28.0,61.2,433
1,"A+B+C, no definitions (02)",59,45.8,37.0,66.4,0
2,A+B + definitions (before rescue),52,45.6,22.5,61.4,580
3,A+B+C + definitions = F,61,51.8,36.1,69.3,0


## 5. Which true categories improved?
For each true category, we compare the share of its texts that land in the right cluster (same Hungarian matching as ACC) without and with definitions. This shows *where* definitions help, e.g. on intents that differ by one action word, or on factual questions (`qa`).

In [6]:
def correct_per_text(preds):
    with contextlib.redirect_stdout(io.StringIO()):
        yt, _ = ev.convert_label_to_ids(y_true_raw); yp, _ = ev.convert_label_to_ids(preds)
    mapping = dict(ev.hungray_aligment(yt, yp)[0])
    return np.array([mapping.get(p, -1) == t for p, t in zip(yp, yt)])

c02, cF = correct_per_text(preds02), correct_per_text(predsF)
y = np.array(y_true_raw)
per_cat = pd.DataFrame([{"true category": cat, "texts": int((y == cat).sum()),
                         "acc without definitions": round(100 * c02[y == cat].mean(), 1),
                         "acc with definitions": round(100 * cF[y == cat].mean(), 1)}
                        for cat in sorted(set(y_true_raw))])
per_cat["change"] = (per_cat["acc with definitions"] - per_cat["acc without definitions"]).round(1)
per_cat = per_cat.sort_values("change", ascending=False)
print("Most improved categories:"); display(per_cat.head(10))
print("Most worsened categories:"); display(per_cat.tail(5))

Most improved categories:


,true category,texts,acc without definitions,acc with definitions,change
55,transport taxi,23,0.0,100.0,100.0
26,iot hue lightup,27,0.0,92.6,92.6
22,iot hue lightchange,36,2.8,69.4,66.6
18,general joke,19,31.6,89.5,57.9
15,email querycontact,26,7.7,65.4,57.7
51,social query,25,28.0,84.0,56.0
53,takeaway query,35,11.4,62.9,51.5
16,email sendemail,114,14.9,64.9,50.0
30,lists query,51,15.7,60.8,45.1
41,play radio,72,13.9,58.3,44.4


Most worsened categories:


,true category,texts,acc without definitions,acc with definitions,change
29,lists createoradd,39,59.0,43.6,-15.4
58,weather query,156,73.7,44.9,-28.8
10,cooking recipe,72,75.0,44.4,-30.6
52,takeaway order,22,77.3,4.5,-72.8
17,general greet,1,100.0,0.0,-100.0


## 6. Save the result
Saved to `results/results.csv` as method **F**, so notebook 04 can show it next to A–E.

In [7]:
s, k = score(predsF)
row = {"method": METHOD_F, "dataset": DATA, "model": MODEL, "n_texts": len(texts),
       "true_clusters": len(set(y_true_raw)), "predicted_clusters": k,
       "ACC": s["ACC"] / 100, "ARI": s["ARI"] / 100, "NMI": s["NMI"] / 100, "unsuccessful": 0,
       "llm_calls": math.ceil(len(texts) / 15) + len(labels) + len(texts),   # label generation + definitions + classification
       "time_min": round(time_classify, 1) if time_classify > 0.5 else np.nan,
       "threshold": THRESHOLD, "date": time.strftime("%Y-%m-%d %H:%M")}
res = pd.read_csv("./results/results.csv")
res = res[~((res.method == METHOD_F) & (res.dataset == DATA) & (res.model == MODEL) & (res.threshold == THRESHOLD))]
res = pd.concat([res, pd.DataFrame([row])], ignore_index=True)
res.to_csv("./results/results.csv", index=False)
print("Saved:", row)

Saved: {'method': 'F (A+B+C + label definitions)', 'dataset': 'massive_intent', 'model': 'qwen2.5:7b', 'n_texts': 2974, 'true_clusters': 59, 'predicted_clusters': 61, 'ACC': 0.518, 'ARI': 0.361, 'NMI': 0.693, 'unsuccessful': 0, 'llm_calls': 3234, 'time_min': 119.0, 'threshold': 0.4, 'date': '2026-11-13 15:59'}


## Findings: Massive-I, Qwen 2.5 7B, t = 0.4

| Variant | Clusters (true 59) | ACC | NMI | ARI | Failed before rescue |
|---|---|---|---|---|---|
| A+B+C, no definitions (02) | 59 | 45.8 | 66.4 | 37.0 | 433 |
| F: A+B+C + definitions | 61 | 51.8 | 69.3 | 36.1 | 580 |

1. **Definitions help:** ACC +6.0, NMI +2.9, ARI −0.9 (about the same). Our method is now level with K-means (silhouette K) on Massive-I: ACC 51.8 vs 52.2, NMI 69.3 vs 69.2.
2. **Where they help:** intents that differ by one action word, as expected: `iot hue lightup` 0% → 93%, `iot hue lightchange` 3% → 69%, `email querycontact` 8% → 65%, `email sendemail` 15% → 65%, `lists query` 16% → 61%, `takeaway query` 11% → 63%.
3. **Gap to GPT-3.5 recovered** (from the original method): ACC 48%, NMI 67%, ARI 58%.
4. **Cost:** prompts grew to ~2,900 tokens (window 8,192, nothing cut), and classification took 119 min instead of 10.6 (11× slower). More answers needed rescue (580 vs 433), because with long prompts the model copies the exact label name less often.
5. **Definition quality:** mostly accurate; one overlap (`music query` was defined to include playing music, overlapping with `play music`). Definitions depend on the example texts the model saw.
6. **Next:** combine with method E (LLM on a 10–20% subset + local classifier) to keep the gain at a fraction of the cost.
7. **Where definitions hurt:** categories whose label had a near-duplicate left after the merge: `weather query` (156 texts) 73.7% → 44.9% (split with `qa_weather`), `cooking recipe` 75.0% → 44.4% (with `qa_cooking`), `takeaway order` 77.3% → 4.5% (with `takeaway query` / `food order`), `lists createoradd` 59.0% → 43.6% (with `todo list` / `grocery list`). The definitions describe each label as distinct, so the model spreads one true category across two labels. Because `weather query` is the largest category, this cancels the ARI gains from the improved intents.
8. **Conclusion:** definitions separate labels well, both the ones that should be separate (`lightup` vs `lightoff`) and the near-duplicates that should not. A possible fix: merge labels whose *definitions* are near-identical before classifying.